# Register a custom parser

This tutorial shows the smallest viable extension point for third-party parsing.
The parser registry accepts either callable functions or parser classes exposing a
`.parse()` method.


## Function-based parser registration

Here we create a small custom parser that reuses the packaged IOT example. In a
real project, this function would read from an external source and return a fully
initialized `Dataset`.


In [ ]:
import mario

from mario.model import Dataset
from mario.parsers import ParserRegistry, register_parser

registry = ParserRegistry()


@register_parser("test_iot", registry=registry)
def parse_test_iot(**kwargs):
    dataset = Dataset.from_database(mario.load_test("IOT"))
    if kwargs.get("name") is not None:
        dataset.metadata.name = kwargs["name"]
    return dataset


In [ ]:
registry.names()


In [ ]:
dataset = registry.parse("test_iot", name="Custom parser demo")

print(dataset.metadata.name)
print(dataset.table_kind.value)
print(dataset.list_blocks())


## The returned object is a normal `Dataset`

Once the parser has returned a `Dataset`, everything else is standard MARIO 2
behavior: storage, scenarios and compute do not care where the data came from.


In [ ]:
w = dataset.compute("w")
w.shape


## Notes for real parser implementations

- use a local `ParserRegistry` during development if you want isolation;
- register on the default global registry only when the parser is meant to be public;
- return `Dataset` objects with correct metadata, indexes and units;
- keep parser-specific logic in the parser layer and leave compute rules to `mario.compute`.
